In [ ]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from tqdm.auto import tqdm
from itertools import product

import sys
sys.path.insert(0, str(Path.cwd().resolve().parents[1] / '2_Propensities'))
sys.path.insert(0, str(Path.cwd().resolve().parents[1] / '4_Baselines' / '4.2_OutcomeModel'))
import MF_class as MF
import OM_class as OM

np.random.seed(42)
if np.random.choice(np.arange(1000)) != 102:
    raise ValueError("Random seed is not set correctly.")

### Score function

$$
s(u,i,j,x) = [p_u; q_i]^{\top} (W + x\Delta) q_j + \alpha_u b_u + \alpha_i b_i + \alpha_j b_j + \alpha_g 
$$

where:

- $u$ : user index  
- $i$ : treatment item  
- $j$ : outcome item
- $x \in \{0,1\}$ : indicator for $u$ interacting with $i$ **and** not intercating with $j$ *before* $i$

- $p_u, q_i, q_j  \in \mathbb{R}^d$ : embedding vectors of user $u$, item $i$, item $j$

- $[p_u; q_i] \in \mathbb{R}^{2d}$ : concatenation of the user and treatment item embeddings

- $W \in \mathbb{R}^{2d \times d}$ : baseline pathway matrix mapping $(u,i)$ to item $j$  
- $\Delta \in \mathbb{R}^{2d \times d}$ : incremental pathway effect under treatment ($x=1$)

- $b_u, b_i, b_j \in \mathbb{R}$ : user, treatment item, and outcome item bias terms

- $\alpha_u, \alpha_i, \alpha_j \in \mathbb{R}$ : scaling coefficients for the bias terms

- $\alpha_g \in \mathbb{R}$ : global bias

- $s(u,i,j,x)$ : predicted logit score for the interaction.

---

### Training loss

$$
\mathcal{L}
= - \sum_n \left[
y_n \log \sigma(s_n)
+ (1 - y_n)\log(1 - \sigma(s_n))
\right]
$$

where:

- $n$ : index of a training observation  
- $y_n \in \{0,1\}$ : observed label (interaction with $j$, occurred or not)  
- $s_n$ : model logit score for observation $n$  
- $\sigma(z) = \frac{1}{1 + e^{-z}}$ : logistic sigmoid function  
- $\mathcal{L}$ : binary cross-entropy loss.

# 1. Load Dataset

In [2]:
base_artifacts = Path.cwd().resolve().parents[2] / 'CausalI2I_artifacts'
data_path = base_artifacts / 'Datasets' / 'Simulation'

In [3]:
om_train_data = pd.read_csv(data_path / 'om_train.csv')
om_test_data  = pd.read_csv(data_path / 'om_test.csv')

# 2. Load MF Embeddings

In [6]:
MF_model = MF.MatrixFactorizationTorch(
    n_users=6040,
    n_items=3952,
    n_factors=20,
)

model_path = base_artifacts / 'Propensity_Models'
model_name = f'MF_simulation'
MF_model.load(path=model_path / (model_name + '.pt'))

user_embeddings = MF_model.P.detach().numpy()[:-1]
item_embeddings = MF_model.Q.detach().numpy()[:-1]
user_bias = MF_model.b_u.detach().numpy()[:-1]
item_bias = MF_model.b_i.detach().numpy()[:-1]

Loaded model summary:
Model:                      MatrixFactorizationTorch
Number of users:            6040
Number of items:            3952
Number of factors:          20
Learning rate:              0.002
Weight decay:               1e-07
Positive weight:            1
Batch size:                 32768
Number of epochs:           50
Device:                     cuda:0
Use AMP:                    True
Timestamp:                  2026-04-11 14:29:10


# 3. Train Outcome Model

In [7]:
model = OM.OutcomeModel(
    user_embeddings = user_embeddings,
    item_embeddings = item_embeddings,
    user_bias = user_bias,
    item_bias = item_bias,
    loss = 'bce'
)

model.fit(
    df_train=om_train_data,
    df_valid=om_test_data,
    lr=2e-4,
    weight_decay=1e-4,
    epochs=20,
    batch_size=2**13,
)

/home/gouni/CausalI2I/4_Baselines/4.3_OutcomeModel/OM_class.py:247: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and is_cuda))


Loss: BCE
Epoch  ||- - - - - - - - Train - - - - - - - -||- - - - - - Validation - - - - - - - || Epoch's | COS θ | Time     
Number || Loss   | L-POS   | L-NEG   | MPR    || Loss   | L-POS   | L-NEG   | MPR    || Change  |       | Elapsed  
=======||========|=========|=========|========||========|=========|=========|========||=========|=======|==========
    1  || 0.6373 |  0.8874 |  0.3872 | 0.7307 || 0.5925 |  0.7117 |  0.4641 | 0.7493 ||  1.1513 | None  |    10.9s
    2  || 0.5653 |  0.6679 |  0.4627 | 0.7671 || 0.5621 |  0.6349 |  0.4836 | 0.7689 ||  0.6543 | 0.604 |    21.5s
    3  || 0.5427 |  0.6185 |  0.4670 | 0.7823 || 0.5465 |  0.6005 |  0.4883 | 0.7786 ||  0.4638 | 0.862 |    32.1s
    4  || 0.5274 |  0.5907 |  0.4642 | 0.7922 || 0.5350 |  0.5796 |  0.4870 | 0.7856 ||  0.3815 | 0.933 |    42.6s
    5  || 0.5150 |  0.5696 |  0.4604 | 0.8001 || 0.5258 |  0.5688 |  0.4796 | 0.7917 ||  0.3425 | 0.961 |    53.0s
    6  || 0.5044 |  0.5532 |  0.4555 | 0.8070 || 0.5180 |  0.5543 |

## Save Model

In [8]:
model_path = base_artifacts / 'Outcome_Models' / f'OM_simulation.pt'

In [9]:
model.save(model_path, note="none")

## Load Model

In [10]:
loaded_model = OM.OutcomeModel(
    user_embeddings = user_embeddings,
    item_embeddings = item_embeddings,
    user_bias = user_bias,
    item_bias = item_bias,
    loss = 'bce'
)

loaded_model.load(model_path)

Loaded OutcomeModel summary:
Model:             OutcomeModel
Embedding dim:     20
Loss:              BCE
Learning rate:     0.0002
Weight decay:      0.0001
Batch size:        8192
Epochs:            20
Use AMP:           True
Timestamp:         2026-04-13 14:07:22
Note:              none
